In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

patients_df = pd.read_csv("patients.csv")
visits_df   = pd.read_csv("visits.csv")
billing_df  = pd.read_csv("billing.csv")

In [ ]:
import sqlite3

conn = sqlite3.connect("hospital_system.db")
cursor = conn.cursor()
conn.execute("PRAGMA foreign_keys = ON;")

In [ ]:
# Merge visits with patients
visits_patients = visits_df.merge(patients_df, on="patient_id", how="left")

# Merge billing
full_df = visits_patients.merge(billing_df, on="visit_id", how="left")

# Quick check
full_df.head()

In [ ]:
missing_summary = full_df[['approved_amount', 'payment_days', 'length_of_stay_hours']].isnull().sum()
missing_summary_percentage = (missing_summary / len(full_df) * 100).round(2)
print(pd.concat([missing_summary, missing_summary_percentage], axis=1, keys=['Missing Count', '% Missing']))

In [ ]:
# Distribution Analysis

import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of visit_type
sns.countplot(data=full_df, x='visit_type')
plt.title('Visit Type Distribution')
plt.show()

# Distribution by department
plt.figure(figsize=(12,6))
sns.countplot(data=full_df, x='department', order=full_df['department'].value_counts().index)
plt.title('Visits by Department')
plt.xticks(rotation=45)
plt.show()

# Distribution by insurance_provider
plt.figure(figsize=(12,6))
sns.countplot(data=full_df, y='insurance_provider', order=full_df['insurance_provider'].value_counts().index)
plt.title('Visits by Insurance Provider')
plt.show()

# Distribution by city
plt.figure(figsize=(12,6))
sns.countplot(data=full_df, x='city', order=full_df['city'].value_counts().index)
plt.title('Visits by City')
plt.xticks(rotation=45)
plt.show()

In [ ]:
numeric_cols = ['billed_amount', 'payment_days', 'length_of_stay_hours']

for col in numeric_cols:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=full_df[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

    # Optional: identify extreme outliers
    q1 = full_df[col].quantile(0.25)
    q3 = full_df[col].quantile(0.75)
    iqr = q3 - q1
    outliers = full_df[(full_df[col] < q1 - 1.5*iqr) | (full_df[col] > q3 + 1.5*iqr)]
    print(f"{col}: {len(outliers)} outliers")

In [ ]:
import numpy as np

# Visit frequency per patient
visit_freq = visits_df.groupby('patient_id').size().reset_index(name='visit_frequency')
full_df = full_df.merge(visit_freq, on='patient_id', how='left')

# Avg length of stay per patient
avg_los = visits_df.groupby('patient_id')['length_of_stay_hours'].mean().reset_index(name='avg_length_of_stay')
full_df = full_df.merge(avg_los, on='patient_id', how='left')

# Provider rejection rate
rejection_rate = full_df.groupby('insurance_provider')['claim_status'].apply(
    lambda x: (x=='Rejected').mean()
).reset_index(name='provider_rejection_rate')
full_df = full_df.merge(rejection_rate, on='insurance_provider', how='left')

# Days since registration
full_df['registration_date'] = pd.to_datetime(full_df['registration_date'])
full_df['visit_date'] = pd.to_datetime(full_df['visit_date'])
full_df['days_since_registration'] = (full_df['visit_date'] - full_df['registration_date']).dt.days

# Time-based features
full_df['visit_month'] = full_df['visit_date'].dt.month
full_df['visit_weekday'] = full_df['visit_date'].dt.weekday
full_df['visit_hour'] = full_df['visit_date'].dt.hour

In [ ]:
full_df.to_csv("model_table.csv", index=False)